In [ ]:
!git clone https://github.com/vngbthang/ie403-Ecom-MultiTask-Complaint-Detection.git
%cd ie403-Ecom-MultiTask-Complaint-Detection

In [ ]:
!git pull

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -r requirements.txt
!pip install -q seqeval torchcrf pytorch-crf

In [ ]:
!pip install -q TorchCRF

In [ ]:
from pathlib import Path

paths = [
    "data/processed/ner_train.json",
    "data/processed/ner_test.json",
    "data/processed/shopee_mapped.csv",
    "src/training/train_phobert_ner.py",
    "src/training/train_phobert_crf_ner.py",
    "src/training/train_multitask.py",
]

for p in paths:
    print(p, "OK" if Path(p).exists() else "MISSING")

In [ ]:
import torch
import transformers
import pandas as pd
import sklearn
import seqeval

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("seqeval OK")

In [ ]:
try:
    from torchcrf import CRF
    print("torchcrf OK")
except Exception as e:
    print("torchcrf FAIL:", e)

try:
    from TorchCRF import CRF
    print("TorchCRF OK")
except Exception as e:
    print("TorchCRF FAIL:", e)

In [ ]:
!python src/training/train_phobert_ner.py \
  --train-json data/processed/ner_train.json \
  --test-json data/processed/ner_test.json \
  --epochs 1 \
  --batch-size 8 \
  --output-dir outputs/metrics/phobert_ner_single_task

In [ ]:
!find src/models -maxdepth 2 -type f -print
!find src/training -maxdepth 2 -type f -print

In [ ]:
!python src/training/train_phobert_crf_ner.py \
  --train-json data/processed/ner_train.json \
  --test-json data/processed/ner_test.json \
  --epochs 1 \
  --batch-size 8 \
  --output-dir outputs/metrics/phobert_crf_ner

In [ ]:
!python src/training/train_multitask.py --help

In [ ]:
from pathlib import Path

file_path = Path("src/training/train_multitask.py")
text = file_path.read_text(encoding="utf-8")

if "from pathlib import Path" not in text:
    lines = text.splitlines()
    insert_idx = 0
    for i, line in enumerate(lines):
        if line.startswith("import ") or line.startswith("from "):
            insert_idx = i + 1
    lines.insert(insert_idx, "from pathlib import Path")
    file_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Patched train_multitask.py")

In [ ]:
from pathlib import Path

file_path = Path("src/training/train_multitask.py")
text = file_path.read_text(encoding="utf-8")

# Xóa mọi dòng import Path cũ nếu có, tránh trùng/lỗi vị trí
lines = [line for line in text.splitlines() if line.strip() != "from pathlib import Path"]

# Chèn chắc chắn ở đầu file
lines.insert(0, "from pathlib import Path")

file_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("First 20 lines:")
print("\n".join(file_path.read_text(encoding="utf-8").splitlines()[:20]))

In [ ]:
!python -m py_compile src/training/train_multitask.py

In [ ]:
!python src/training/train_multitask.py \
  --cls-path data/processed/shopee_mapped.csv \
  --ner-train-path data/processed/ner_train.json \
  --ner-test-path data/processed/ner_test.json \
  --epochs 1 \
  --batch-size 8 \
  --alpha 1.0 \
  --max-steps-per-epoch 50 \
  --output-dir outputs/metrics/multitask_smoke_test

In [ ]:
!python src/evaluation/collect_results.py \
  --metrics-dir outputs/metrics \
  --output-dir outputs/metrics/summary

In [ ]:
!find outputs -maxdepth 4 -type f | sort

In [ ]:
!find outputs/metrics/multitask_smoke_test -maxdepth 4 -type f | sort

In [ ]:
!python src/evaluation/collect_results.py \
  --metrics-dir outputs/metrics \
  --output-dir outputs/metrics/summary

In [ ]:
import pandas as pd
from pathlib import Path

for file in [
    "outputs/metrics/summary/classification_summary.csv",
    "outputs/metrics/summary/ner_summary.csv",
    "outputs/metrics/summary/ablation_summary.csv",
]:
    if Path(file).exists():
        print("\n===", file, "===")
        display(pd.read_csv(file))

In [ ]:
import json
from pathlib import Path

p = Path("outputs/metrics/multitask_smoke_test/ner_metrics_epoch1.json")
data = json.loads(p.read_text(encoding="utf-8"))

print(data.keys())
print(json.dumps(data, indent=2, ensure_ascii=False))

In [ ]:
import json
from pathlib import Path

p = Path("outputs/metrics/multitask_smoke_test/ner_metrics_epoch1.json")
data = json.loads(p.read_text(encoding="utf-8"))

data["model"] = "Multi-task PhoBERT + CRF"
data["dataset"] = "NER"
data["epoch"] = 1
data["alpha"] = 1.0
data["note"] = "Smoke test with max_steps_per_epoch=50"

p.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

print("Updated:", p)

In [ ]:
!python src/evaluation/collect_results.py \
  --metrics-dir outputs/metrics \
  --output-dir outputs/metrics/summary

In [ ]:
import pandas as pd
from pathlib import Path

p = Path("outputs/metrics/summary/ner_summary.csv")
df = pd.read_csv(p)

df.loc[
    df["source_file"].str.contains("multitask_smoke_test", na=False),
    "model"
] = "Multi-task PhoBERT + CRF"

df.to_csv(p, index=False, encoding="utf-8-sig")
display(df)

In [ ]:
!zip -r smoke_test_outputs.zip outputs

In [ ]:
!python src/training/train_phobert_ner.py \
  --train-json data/processed/ner_train.json \
  --test-json data/processed/ner_test.json \
  --epochs 5 \
  --batch-size 8 \
  --output-dir outputs/metrics/phobert_ner_single_task_full

In [ ]:
!python src/training/train_phobert_crf_ner.py \
  --train-json data/processed/ner_train.json \
  --test-json data/processed/ner_test.json \
  --epochs 5 \
  --batch-size 8 \
  --output-dir outputs/metrics/phobert_crf_ner_full

In [ ]:
!du -h -d 3 outputs | sort -h

In [ ]:
!rm -f outputs/metrics/phobert_ner_single_task/checkpoints/*.pt
!rm -f outputs/metrics/phobert_crf_ner/checkpoints/*.pt
!rm -f outputs/metrics/phobert_ner_single_task_full/checkpoints/*.pt
!rm -f outputs/metrics/phobert_crf_ner_full/checkpoints/*.pt
!rm -f outputs/metrics/multitask_smoke_test/*.pt
!rm -f outputs/metrics/multitask_smoke_test/checkpoints/*.pt

In [ ]:
!python src/training/train_multitask.py \
  --cls-path data/processed/shopee_mapped.csv \
  --ner-train-path data/processed/ner_train.json \
  --ner-test-path data/processed/ner_test.json \
  --epochs 3 \
  --batch-size 8 \
  --alpha 2.0 \
  --only-ner-matched \
  --output-dir outputs/metrics/multitask_alpha2_ner_matched

In [ ]:
!rm -f outputs/metrics/multitask_alpha2_ner_matched/checkpoint_epoch_1.pt
!rm -f outputs/metrics/multitask_alpha2_ner_matched/checkpoint_epoch_2.pt

In [ ]:
!python src/training/train_multitask.py \
  --cls-path data/processed/shopee_mapped.csv \
  --ner-train-path data/processed/ner_train.json \
  --ner-test-path data/processed/ner_test.json \
  --epochs 3 \
  --batch-size 8 \
  --alpha 1.0 \
  --only-ner-matched \
  --output-dir outputs/metrics/multitask_alpha1_ner_matched

In [ ]:
!python src/evaluation/collect_results.py \
  --metrics-dir outputs/metrics \
  --output-dir outputs/metrics/summary

In [ ]:
import pandas as pd
from pathlib import Path

p = Path("outputs/metrics/summary/ner_summary.csv")
df = pd.read_csv(p)

df.loc[df["source_file"].str.contains("multitask_smoke_test", na=False), "model"] = "Multi-task PhoBERT + CRF (smoke test)"
df.loc[df["source_file"].str.contains("multitask_alpha2_ner_matched", na=False), "model"] = "Multi-task PhoBERT + CRF (alpha=2.0, NER-matched)"
df.loc[df["source_file"].str.contains("multitask_alpha1_ner_matched", na=False), "model"] = "Multi-task PhoBERT + CRF (alpha=1.0, NER-matched)"

df.to_csv(p, index=False, encoding="utf-8-sig")
display(df)

In [ ]:
!zip -r final_training_outputs_light.zip outputs -x "*.pt"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv("outputs/metrics/summary/ner_summary.csv")

# Chỉ lấy epoch 3 của hai cấu hình full, bỏ smoke test
plot_df = df[
    (df["epoch"] == 3) &
    (df["model"].str.contains("NER-matched", na=False))
].copy()

plot_df["short_model"] = plot_df["model"].replace({
    "Multi-task PhoBERT + CRF (alpha=2.0, NER-matched)": "MTL alpha=2.0",
    "Multi-task PhoBERT + CRF (alpha=1.0, NER-matched)": "MTL alpha=1.0",
})

out_dir = Path("outputs/figures")
out_dir.mkdir(parents=True, exist_ok=True)

# Entity F1
plt.figure(figsize=(8, 5))
bars = plt.bar(plot_df["short_model"], plot_df["entity_f1"])
plt.ylim(0, 0.4)
plt.ylabel("Entity F1")
plt.xlabel("Model")
plt.title("Entity-F1 Comparison — Multi-task NER")
for bar, val in zip(bars, plot_df["entity_f1"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.4f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(out_dir / "ner_entity_f1_comparison.png", dpi=150)
plt.show()

# Token F1
plt.figure(figsize=(8, 5))
bars = plt.bar(plot_df["short_model"], plot_df["token_f1"])
plt.ylim(0, 0.75)
plt.ylabel("Token F1 Macro")
plt.xlabel("Model")
plt.title("Token-F1 Comparison — Multi-task NER")
for bar, val in zip(bars, plot_df["token_f1"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.4f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(out_dir / "ner_token_f1_comparison.png", dpi=150)
plt.show()

In [ ]:
!zip -r final_training_outputs_light2.zip outputs -x "*.pt"